# SDPA 的分块与多核负载均衡


## 1. 维度分析：切什么、不切什么

本课程 Q/K/V 为 [B,S,H] 单头；H 是特征维，不是独立 head 轴。

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">维度</th><th style="text-align:left;">教学基线</th><th style="text-align:left;">理由</th></tr>
<tr><td>B</td><td>核间分配</td><td>不同 batch 相互独立</td></tr>
<tr><td>S_q</td><td>核间分配 Q 块</td><td>每个 Q 块独立维护 m/l/acc</td></tr>
<tr><td>S_kv</td><td>核内分块</td><td>基线按序更新每行在线状态，避免额外归并</td></tr>
<tr><td>H</td><td>同一任务内处理</td><td>QKᵀ 的归约维，不能按“独立 head”直接分核</td></tr>
</table>

基线规则：**B/S_q 核间、S_kv 核内**。Split-KV 是可行的进阶方案，但需按各分片最大值重新缩放并合并 l/acc，不能直接相加或覆盖输出。


## 2. 教学版切分方案

以 [1, 2048, 64]（单 batch 单头，聚焦核心循环）为例、8 核：

1. **任务**：Q 按 256 行一块 → 8 个 Q 块任务，任务编号连续分配（第 4 章模板）——负载天然均衡（causal 时见第 3 点）；
2. **核内**：K/V 按 256 列一块（kv_block），flash 循环逐块推进；Q 块与 K 块的 Tile 尺寸（q_tile=256, kv_tile=256, H=64）决定片上占用；
3. **跳过完全遮蔽块后的 causal 负载倾斜**：第 i 个 Q 块只需处理前 i+1 个 K 块，任务量呈三角分布。对策：
   - 简单版：实现整块跳过后，任务数不多时先接受倾斜，并记录在性能分析中；
   - 改进版：优先缩小 Q 块增加任务数，按各 Q 块可见 KV 工作量分配；只有任务数足够时重排才有均衡空间。进一步 Split-KV 需额外 workspace 与稳定归并，作为进阶研究，不是主线必须项。

注意：10.04 的 GM 中转参考目前仍对所有 selected 块执行 QK 和 PV，只把不可见位置的概率置 0，尚未跳过完全遮蔽块。因此，该参考的 causal 路径不具备这里推演的三角计算量，也不能直接用它测量跳过后的负载倾斜。


## 3. causal 任务三角的直观图

8 个 Q 块 × 8 个 K 块的可见性矩阵（✓=需计算，·=跳过）：

```text
K块:    0    1    2    3    4    5    6    7
Q块 0   ✓    ·    ·    ·    ·    ·    ·    ·
Q块 1   ✓    ✓    ·    ·    ·    ·    ·    ·
Q块 2   ✓    ✓    ✓    ·    ·    ·    ·    ·
Q块 3   ✓    ✓    ✓    ✓    ·    ·    ·    ·
Q块 4   ✓    ✓    ✓    ✓    ✓    ·    ·    ·
Q块 5   ✓    ✓    ✓    ✓    ✓    ✓    ·    ·
Q块 6   ✓    ✓    ✓    ✓    ✓    ✓    ✓    ·
Q块 7   ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
```

计算量 = 三角形内的块数 = 8×9/2 = 36（vs 全量 64）——**只有实现实际跳过完全遮蔽块时才减少这些块的计算；对角块仍执行矩阵乘**。连续分配 8 个任务给 8 核时，核 0 拿 1 块、核 7 拿 8 块——8 倍倾斜。本例 8 个任务恰好分给 8 核，仅重排任务不会降低最重任务耗时；需增加 Q 任务数或评估 Split-KV。

**跨边界块的放大视图**（Q 块 2 × K 块 2，块尺寸 8 示意）：行 16~23 的可见列 = 各自行号+1 → 行16 见列0~16，行23 见列0~23 → 块内上三角不可见，须按全局 token 位置把这些 logits 置 -inf；valid_shape 只表示矩形，不能表达三角区域。


## 4. 三档规模的预算对照（培养"一眼估"能力）

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">规模</th><th style="text-align:left;">q_tiles×kv_tiles（256 块切）</th><th style="text-align:left;">全量 vs causal 计算量</th></tr>
<tr><td>S=1024</td><td>4×4</td><td>16 vs 10 块</td></tr>
<tr><td>S=2048</td><td>8×8</td><td>64 vs 36 块</td></tr>
<tr><td>S=4096</td><td>16×16</td><td>256 vs 136 块</td></tr>
</table>

规律：causal 系数 ≈ (n+1)/2n → 逼近 1/2；**单个 Q 任务的工作量从首块到末块逐渐增大**；映射后的核间不均衡还取决于任务数、核数与分配策略；S 增大后，均衡调度是否更有收益仍需实测。


## 5. 片上预算（Tile 规划）

按第 2 章方法估算（FP16 输入、FP32 累加）：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">Tile</th><th style="text-align:left;">尺寸</th><th style="text-align:left;">Buffer</th><th style="text-align:left;">字节（示例）</th></tr>
<tr><td>Q 块</td><td>[256, 64] FP16</td><td>L1</td><td>32KB</td></tr>
<tr><td>K 块 / V 块</td><td>[256, 64] FP16</td><td>L1</td><td>32KB（各）</td></tr>
<tr><td>s 块 = Q@K^T</td><td>[256, 256] FP32</td><td>L0C</td><td>256KB</td></tr>
<tr><td>p 块（exp 后）</td><td>[256, 256] FP16</td><td>UB</td><td>128KB</td></tr>
<tr><td>acc</td><td>[256, 64] FP32</td><td>UB</td><td>64KB</td></tr>
<tr><td>m / l</td><td>[256, 1] FP32</td><td>UB</td><td>各 1KB</td></tr>
</table>

L1 双缓冲（K/V 各两槽）+ L0C 单块 + UB 若干——**预算表先于代码**，地址按此规划（第 2 章纪律）。本表只列出逻辑张量的字节数，**不能直接作为完整分配方案**：MIX 的两个 AIV 各有自己的 UB，需按行拆分；还漏了 s 的 UB 副本、FP32 p、归约临时、PV 接收区、转换和槽数。必须逐核统计物理占用。官方 tail/TileGroup FA 示例使用 TS=TKV=128、每 AIV 64 行，并显式检查 UB 预算，可作为缩小 Tile 的起点；不能把 p 转 FP16 后就宣称满足容量和精度。


### 完整参考的物理预算：128×128，每 AIV 64 行

`src/sdpa_reference.py` 固定 Q_BLOCK=KV_BLOCK=128，H=64/128。以较大的 H=128 计：L1 四块 Q/K/P/V 各最多 32 KiB，共 128 KiB；L0A 两块共 64 KiB，L0B 两块共 64 KiB；L0C 的 QK 与 PV 各 64 KiB，共 128 KiB。各空间独立，地址从 0 重用不表示跨空间别名。

每个 AIV 的 UB 按地址依次保留：score 32 KiB、p_FP16 16 KiB、PV 32 KiB、acc 32 KiB、输出 FP16 16 KiB、归约临时 32 KiB；五组行状态各 256 B，总范围到 0x28500（161.25 KiB）。每组行状态的 DN/RM 视图共享同一存储，容量只计一次。两个视图使用相同的 mutex ID，使 auto_mutex 能识别对同一存储的访问依赖。此基线只有单槽握手，不能据此声称已有双缓冲重叠。

每个逻辑 MIX 组独占 GM 的 score[128,128]、p[128,128]、PV[128,H]，H=128 时共 160 KiB。核数为 C 时共 C×160 KiB，与 S² 无关；多个 batch/Q 任务通过同一组的顺序循环复用。4 组处理 B=2、S=1024 时有 16 个任务，每组 4 个，输出区间不重叠。


## 6. 课后练习

1. （填空）SDPA 核间切分黄金律：B/S_q 做 ______，S_kv 只做 ______；原因是 ______。
2. （计算）causal、S_q=S_kv=2048、q_block=kv_block=256：8 个 Q 块任务各自要处理几个 K 块？最忙与最闲核的任务量差多少倍？
3. （实践）把片上预算表改算 H=128 的版本；若假定单个 AIV 可用预算为 256 KiB（仅算术练习，不是设备规格声明），p 块 [256,256] FP16 还放得下吗？给出调整方案（提示：缩小 kv_block）。

In [ ]:
!cat ./answer/10.03_answer.txt

## 7. 本节小结

- 黄金律：**B/S_q 核间、S_kv 核内；H 为归约特征维**（m/l/acc 行状态决定）；
- 实现完全遮蔽块跳过后，causal 会出现三角任务倾斜：先增加 Q 任务并按代价分配，再研究 Split-KV；
- 先列片上预算，再写代码：分别核算 L1、L0A/B/C 和各 AIV 的 UB；多槽或双缓冲要另计容量，当前完整参考仍是单槽串行交接。